In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.base import BaseEstimator, ClassifierMixin

pd.set_option('display.max_columns', 200)
plt.rcParams['figure.dpi'] = 100

df = pd.read_parquet('../data/loans_clean.parquet')

df['dti'] = df['dti'].replace([-1, 999], np.nan).clip(upper=100)
df["credit_history_months"] = df['credit_history_months'].replace(999, np.nan)
df['annual_inc'] = df["annual_inc"].clip(upper=1_000_000)
df['revol_util'] = df['revol_util'].clip(upper=100)

df_val   = df[df["issue_year"] == 2016].copy()
y_val   = df_val['default'].values


print(f"Val loans: {len(df_val):,}")

Val loans: 293,095


In [2]:
df_val_fe = df_val.copy()

def prep_features(df):
    df = df.copy()
    
    redundant = [
        "fico_range_high",
        "funded_amnt",
        "funded_amnt_inv",
        "num_sats",
        "installment",
        "num_rev_tl_bal_gt_0",
    ]

    joint_cols = [c for c in df.columns if c.startswith("sec_app") or c.endswith("_joint")]

    high_cardinality = ["zip_code", "sub_grade"]

    split_cols = ["issue_year"]

    emp_map = {
        "< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4,
        "5 years": 5, "6 years": 6, '7 years': 7, "8 years": 8, "9 years": 9,
        "10+ years": 10
    }
    df["emp_length"] = df["emp_length"].map(emp_map)
    
    cols_to_drop = redundant + joint_cols + high_cardinality + split_cols
    
    df = df.drop(
        columns=[c for c in cols_to_drop if c in df.columns]
    )
    
    return df



df_val_fe = prep_features(df_val_fe)
x_val = df_val_fe.drop(columns=['default'])
x_val['term'] = x_val['term'].str.extract(r'(\d+)').astype(int)

numeric_cols = x_val.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = x_val.select_dtypes(include=['object']).columns.tolist()

x_val[numeric_cols] = x_val[numeric_cols].astype('float64')

print(f"x_val: {x_val.shape}, NaNs (handled by preprocessor): {x_val.isna().sum().sum()}")


x_val: (293095, 79), NaNs (handled by preprocessor): 1113613


In [3]:
champion = joblib.load("../models/champion_lgbm.pkl")


class CalibratedChampionClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, preprocessor, model, calibrator, threshold):
        self.preprocessor = preprocessor
        self.model = model
        self.calibrator = calibrator
        self.threshold = threshold
        self.classes_ = np.array([0, 1])
    def fit(self, X, y):
        return self
    def predict_proba(self,X):
        X_proc = self.preprocessor.transform(X)
        raw = self.model.predict_proba(X_proc)[:, 1]
        cal = self.calibrator.predict(raw)
        return np.column_stack([1 - cal, cal])
    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= self.threshold).astype(int)


champion_clf = CalibratedChampionClassifier(
    preprocessor=champion['preprocessor'],
    model=champion['base_model'],
    calibrator=champion['calibrator'],
    threshold=champion['best_threshold']
)
best_threshold = champion['best_threshold']

all_probs = champion_clf.predict_proba(x_val)[:, 1]
print(f"Champion loaded. Threshold: {best_threshold:.4f}")
print(f"Approved in val: {(all_probs < best_threshold).sum():,}")
print(f"Rejected in val: {(all_probs >= best_threshold).sum():,}")

/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Champion loaded. Threshold: 0.1592
Approved in val: 109,622
Rejected in val: 183,473


In [4]:
# EASY case: P just above threshold (single-feature fixes likely possible)
easy_mask = (
    (all_probs >= 0.16) & (all_probs <= 0.22) &
    (x_val['grade'].isin(['C', 'D'])) &
    (x_val['term'] == 60)
)
easy_idx = int(np.where(easy_mask)[0][0])

# HARD case: P much higher (will need multiple changes)
hard_mask = (
    (all_probs >= 0.30) & (all_probs <= 0.40) &
    (x_val['grade'].isin(['C', 'D'])) &
    (x_val['term'] == 60)
)
hard_idx = int(np.where(hard_mask)[0][0])

for label, idx in [("EASY (P just above threshold)", easy_idx),
                    ("HARD (P much higher)", hard_idx)]:
    r = x_val.iloc[idx]
    print(f"\n=== {label} — borrower idx {idx} ===")
    print(f"  P(default): {all_probs[idx]:.4f}  (threshold: {best_threshold:.4f})")
    print(f"  Loan:     ${r['loan_amnt']:,.0f}, {r['term']:.0f} months, {r['int_rate']:.1f}% rate, grade {r['grade']}")
    print(f"  Borrower: ${r['annual_inc']:,.0f}/yr, DTI {r['dti']:.1f}%, FICO {r['fico_range_low']:.0f}")
    print(f"  Other:    emp_length={r['emp_length']:.0f} yrs, {r['home_ownership']}, {r['purpose']}")


=== EASY (P just above threshold) — borrower idx 1 ===
  P(default): 0.2126  (threshold: 0.1592)
  Loan:     $21,000, 60 months, 14.5% rate, grade C
  Borrower: $80,000/yr, DTI 9.9%, FICO 735
  Other:    emp_length=0 yrs, MORTGAGE, debt_consolidation

=== HARD (P much higher) — borrower idx 19 ===
  P(default): 0.3643  (threshold: 0.1592)
  Loan:     $17,200, 60 months, 14.5% rate, grade C
  Borrower: $73,000/yr, DTI 13.6%, FICO 660
  Other:    emp_length=8 yrs, MORTGAGE, credit_card


In [5]:
def find_single_feature_cfs(query_row, model, feature_specs, threshold, n_steps=100):
    """For each feature, find the smallest change that flips REJECT to APPROVE."""
    results = []
    
    for feature, (low, high) in feature_specs.items():
        original_val = query_row[feature].iloc[0]
        values = np.linspace(low, high, n_steps)
        
        batch = pd.concat([query_row] * n_steps, ignore_index=True)
        batch[feature] = values
        probs = model.predict_proba(batch)[:, 1]
        
        flips = probs < threshold
        if not flips.any():
            continue

        changes = np.abs(values - original_val)
        changes[~flips] = np.inf
        best = np.argmin(changes)
        
        results.append({
            'feature':   feature,
            'original':  original_val,
            'new':       values[best],
            'change':    values[best] - original_val,
            'new_prob':  probs[best]
        })
    
    return results


feature_specs = {
    'loan_amnt':            (1000, 40000),
    'dti':                  (0, 60),
    'annual_inc':           (20000, 500000),
    'fico_range_low':       (600, 850),
}


def print_cfs(cfs, label, original_prob):
    print(f"\n{'='*80}")
    print(f"{label}: P(default)={original_prob:.4f}, REJECTED")
    print(f"{'='*80}")
    if not cfs:
        print("  No single-feature change can flip this rejection.")
        return
    for cf in cfs:
        if cf['feature'] in ['loan_amnt', 'annual_inc']:
            print(f"  {cf['feature']:25}: ${cf['original']:>10,.0f} -> ${cf['new']:>10,.0f}  "
                  f"({cf['change']:+,.0f})  P -> {cf['new_prob']:.4f}")
        else:
            print(f"  {cf['feature']:25}: {cf['original']:>10.2f} -> {cf['new']:>10.2f}  "
                  f"({cf['change']:+.2f})  P -> {cf['new_prob']:.4f}")


easy_cfs = find_single_feature_cfs(x_val.iloc[[easy_idx]], champion_clf, feature_specs, best_threshold)
print_cfs(easy_cfs, f"EASY CASE (idx {easy_idx})", all_probs[easy_idx])

hard_cfs = find_single_feature_cfs(x_val.iloc[[hard_idx]], champion_clf, feature_specs, best_threshold)
print_cfs(hard_cfs, f"HARD CASE (idx {hard_idx})", all_probs[hard_idx])


EASY CASE (idx 1): P(default)=0.2126, REJECTED
  loan_amnt                : $    21,000 -> $     2,576  (-18,424)  P -> 0.1579

HARD CASE (idx 19): P(default)=0.3643, REJECTED
  No single-feature change can flip this rejection.


/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/nachimora

In [6]:
def find_pairwise_cfs(query_row, model, feature_specs, threshold, n_per_feature=15):
    """Find pairs of feature changes that flip rejection. Returns sorted by minimum combined change."""
    features = list(feature_specs.keys())
    results = []
    
    for i, f1 in enumerate(features):
        low1, high1 = feature_specs[f1]
        v1_range = np.linspace(low1, high1, n_per_feature)
        orig1 = query_row[f1].iloc[0]
        
        for j, f2 in enumerate(features):
            if j <= i:
                continue
            low2, high2 = feature_specs[f2]
            v2_range = np.linspace(low2, high2, n_per_feature)
            orig2 = query_row[f2].iloc[0]
            
            v1s, v2s = np.meshgrid(v1_range, v2_range)
            n = v1s.size
            
            batch = pd.concat([query_row] * n, ignore_index=True)
            batch[f1] = v1s.flatten()
            batch[f2] = v2s.flatten()
            
            probs = model.predict_proba(batch)[:, 1]
            flips = probs < threshold
            if not flips.any():
                continue
            
            v1_flat = v1s.flatten()
            v2_flat = v2s.flatten()
            
            norm = (np.abs(v1_flat - orig1) / (high1 - low1) +
                    np.abs(v2_flat - orig2) / (high2 - low2))
            norm[~flips] = np.inf
            best = np.argmin(norm)
            
            results.append({
                'features': (f1, f2),
                'orig':     (orig1, orig2),
                'new':      (v1_flat[best], v2_flat[best]),
                'changes':  (v1_flat[best] - orig1, v2_flat[best] - orig2),
                'norm':     norm[best],
                'new_prob': probs[best],
            })
    
    results.sort(key=lambda x: x['norm'])
    return results


print(f"\n{'='*80}")
print(f"HARD CASE: pairwise counterfactuals (top 5)")
print(f"{'='*80}")
print("Computing pairwise grid search (~30s)...")

pair_cfs = find_pairwise_cfs(x_val.iloc[[hard_idx]], champion_clf, feature_specs, best_threshold)

if not pair_cfs:
    print("\nNo pair of changes flips this rejection. Need 3+ features simultaneously.")
else:
    for i, cf in enumerate(pair_cfs[:5]):
        f1, f2 = cf['features']
        print(f"\nOption {i+1}: change both {f1} and {f2}")
        print(f"  {f1:25}: {cf['orig'][0]:>10.2f} -> {cf['new'][0]:>10.2f}  ({cf['changes'][0]:+.2f})")
        print(f"  {f2:25}: {cf['orig'][1]:>10.2f} -> {cf['new'][1]:>10.2f}  ({cf['changes'][1]:+.2f})")
        print(f"  P(default): {all_probs[hard_idx]:.4f} -> {cf['new_prob']:.4f}")


HARD CASE: pairwise counterfactuals (top 5)
Computing pairwise grid search (~30s)...


/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



No pair of changes flips this rejection. Need 3+ features simultaneously.


/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/nachimorales/Documents/projects/loan-default-fairness/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
